In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
%cd /kaggle/working
!mkdir -p data models utils




/kaggle/working


In [2]:
%%writefile /kaggle/working/requirements.txt
torch
torchvision
numpy
pandas
scikit-learn
opencv-python


Writing /kaggle/working/requirements.txt


In [3]:
!pip install -r /kaggle/working/requirements.txt


In [4]:
%%writefile /kaggle/working/utils/dataset_loader.py
import cv2
import torch
from torch.utils.data import Dataset
import os

class VideoFrameDataset(Dataset):
    def __init__(self, video_paths, labels, transform=None, frames_per_clip=16):
        self.video_paths = video_paths
        self.labels = labels
        self.transform = transform
        self.frames_per_clip = frames_per_clip

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        
        cap = cv2.VideoCapture(video_path)
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            if self.transform:
                frame = self.transform(frame)
            frames.append(frame)
        cap.release()

        if len(frames) >= self.frames_per_clip:
            frames = frames[:self.frames_per_clip]
        else:
            # pad with last frame
            while len(frames) < self.frames_per_clip:
                frames.append(frames[-1])

        video_tensor = torch.stack(frames)
        return video_tensor, torch.tensor(label)


Writing /kaggle/working/utils/dataset_loader.py


In [5]:
%%writefile /kaggle/working/models/resnet_lstm.py
import torch
import torch.nn as nn
import torchvision.models as models

class ResNetLSTM(nn.Module):
    def __init__(self, hidden_size=256, num_classes=2):
        super(ResNetLSTM, self).__init__()
        resnet = models.resnet50(pretrained=True)
        modules = list(resnet.children())[:-1]  # remove final fc
        self.resnet = nn.Sequential(*modules)
        self.lstm = nn.LSTM(2048, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):  # x: [B, T, C, H, W]
        B, T, C, H, W = x.shape
        cnn_embed_seq = []
        for t in range(T):
            with torch.no_grad():
                feat = self.resnet(x[:, t, :, :, :])
            feat = feat.view(B, -1)
            cnn_embed_seq.append(feat)
        cnn_embed_seq = torch.stack(cnn_embed_seq, dim=1)
        lstm_out, _ = self.lstm(cnn_embed_seq)
        lstm_last_out = lstm_out[:, -1, :]
        out = self.fc(lstm_last_out)
        return out


Writing /kaggle/working/models/resnet_lstm.py


In [6]:
import os, json, glob

# path to training set
TRAIN_PATH = "/kaggle/input/deepfake-detection-challenge/train_sample_videos"

# check metadata.json
meta_file = os.path.join(TRAIN_PATH, "metadata.json")

with open(meta_file, "r") as f:
    metadata = json.load(f)

print("Total videos:", len(metadata))
print("Sample:", list(metadata.items())[:5])  # print 5 entries


Total videos: 400
Sample: [('aagfhgtpmv.mp4', {'label': 'FAKE', 'split': 'train', 'original': 'vudstovrck.mp4'}), ('aapnvogymq.mp4', {'label': 'FAKE', 'split': 'train', 'original': 'jdubbvfswz.mp4'}), ('abarnvbtwb.mp4', {'label': 'REAL', 'split': 'train', 'original': None}), ('abofeumbvv.mp4', {'label': 'FAKE', 'split': 'train', 'original': 'atvmxvwyns.mp4'}), ('abqwwspghj.mp4', {'label': 'FAKE', 'split': 'train', 'original': 'qzimuostzz.mp4'})]


In [7]:
video_paths = []
labels = []

for filename, meta in metadata.items():
    video_path = os.path.join(TRAIN_PATH, filename)
    label = 1 if meta["label"] == "FAKE" else 0  # 0=REAL, 1=FAKE
    video_paths.append(video_path)
    labels.append(label)

print("Videos:", len(video_paths))
print("REAL:", labels.count(0), "FAKE:", labels.count(1))


Videos: 400
REAL: 77 FAKE: 323


In [11]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

# ======================
# CONFIG
# ======================
DATA_DIR = "/kaggle/input/deepfake-detection-challenge-frames"
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# DATASET & DATALOADER
# ======================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=train_transforms)

# Class counts
class_counts = [0, 0]
for _, label in dataset.samples:
    class_counts[label] += 1
real_count, fake_count = class_counts
print(f"Class distribution → REAL: {real_count}, FAKE: {fake_count}")

# Weighted loss
total = sum(class_counts)
weights = [total / real_count, total / fake_count]
class_weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# ======================
# MODEL
# ======================
from torchvision import models

model = models.resnet50(pretrained=True)


num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # binary classification
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=LR)

# ======================
# TRAINING LOOP
# ======================
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total_samples = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = correct / total_samples
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

# ======================
# SAVE MODEL
# ======================
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/resnet50_deepfake.pth")
print("Training complete, model saved.")


Class distribution → REAL: 4267, FAKE: 1218
Epoch [1/10] Loss: 0.4696 Acc: 0.7451
Epoch [2/10] Loss: 0.3472 Acc: 0.8115
Epoch [3/10] Loss: 0.2956 Acc: 0.8268
Epoch [4/10] Loss: 0.2764 Acc: 0.8447
Epoch [5/10] Loss: 0.2622 Acc: 0.8583
Epoch [6/10] Loss: 0.2510 Acc: 0.8660
Epoch [7/10] Loss: 0.2265 Acc: 0.8828
Epoch [8/10] Loss: 0.2088 Acc: 0.8901
Epoch [9/10] Loss: 0.1987 Acc: 0.9010
Epoch [10/10] Loss: 0.1846 Acc: 0.9103
Training complete, model saved.


In [14]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# ======================
# CONFIG
# ======================
DATA_DIR = "/kaggle/input/deepfake-detection-challenge-frames"
BATCH_SIZE = 32
NUM_EPOCHS = 10
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# DATASET + DATALOADER
# ======================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2)

# Check class distribution
class_counts = [0] * len(train_dataset.classes)
for _, label in train_dataset:
    class_counts[label] += 1

print("Class counts:", dict(zip(train_dataset.classes, class_counts)))

# Compute class weights for imbalance
weights = torch.tensor([sum(class_counts) / c for c in class_counts], dtype=torch.float).to(DEVICE)
print("Class weights:", weights)

# ======================
# MODEL
# ======================
model = models.resnet50(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # binary classification: real vs fake
model = model.to(DEVICE)

# Loss with class weights
criterion = nn.CrossEntropyLoss(weight=weights)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LR)

# ======================
# TRAIN LOOP
# ======================
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

# ======================
# SAVE MODEL
# ======================
torch.save(model.state_dict(), "/kaggle/working/resnet50_deepfake.pth")
print("Model saved to /kaggle/working/resnet50_deepfake.pth")


Class counts: {'fake': 4267, 'real': 1218}
Class weights: tensor([1.2854, 4.5033], device='cuda:0')
Epoch [1/10] Loss: 0.4534 Acc: 0.7544
Epoch [2/10] Loss: 0.3054 Acc: 0.8272
Epoch [3/10] Loss: 0.2672 Acc: 0.8531
Epoch [4/10] Loss: 0.2437 Acc: 0.8691
Epoch [5/10] Loss: 0.2137 Acc: 0.8972
Epoch [6/10] Loss: 0.1841 Acc: 0.9088
Epoch [7/10] Loss: 0.1613 Acc: 0.9278
Epoch [8/10] Loss: 0.1240 Acc: 0.9468
Epoch [9/10] Loss: 0.1000 Acc: 0.9597
Epoch [10/10] Loss: 0.1072 Acc: 0.9615
Model saved to /kaggle/working/resnet50_deepfake.pth


In [15]:
!ls /kaggle/input/deepfake-detection-challenge
!ls /kaggle/input/deepfake-detection-challenge/train_sample_videos | head

!ls /kaggle/input/deepfake-detection-challenge




sample_submission.csv  test_videos  train_sample_videos
aagfhgtpmv.mp4
aapnvogymq.mp4
abarnvbtwb.mp4
abofeumbvv.mp4
abqwwspghj.mp4
acifjvzvpm.mp4
acqfdwsrhi.mp4
acxnxvbsxk.mp4
acxwigylke.mp4
aczrgyricp.mp4
sample_submission.csv  test_videos  train_sample_videos


In [16]:
# ======================
# Imports
# ======================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

# ======================
# Device
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ======================
# Dataset & Dataloaders
# ======================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# adjust path to your dataset
dataset = ImageFolder(root="/kaggle/input/deepfake-detection-challenge-frames", 
                      transform=transform)

train_size = int(0.7 * len(dataset))
val_size   = int(0.15 * len(dataset))
test_size  = len(dataset) - train_size - val_size

train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)

print("Classes:", dataset.classes)
print(f"Train/Val/Test: {len(train_set)} {len(val_set)} {len(test_set)}")

# ======================
# Model (ResNet50)
# ======================
from torchvision.models import resnet50
model = resnet50(pretrained=True)  # old API for Kaggle env

# replace FC layer for 2 classes (fake/real)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# ======================
# Loss & Optimizer
# ======================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# ======================
# Training Loop
# ======================
EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%")

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    print(f"Validation Acc: {val_acc:.2f}%")

# ======================
# Test Evaluation
# ======================
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")


Using device: cuda
Classes: ['fake', 'real']
Train/Val/Test: 3839 822 824
Epoch 1/5, Loss: 0.3984, Train Acc: 80.57%
Validation Acc: 87.23%
Epoch 2/5, Loss: 0.2652, Train Acc: 87.60%
Validation Acc: 87.35%
Epoch 3/5, Loss: 0.2182, Train Acc: 90.28%
Validation Acc: 84.91%
Epoch 4/5, Loss: 0.1795, Train Acc: 91.77%
Validation Acc: 88.20%
Epoch 5/5, Loss: 0.1549, Train Acc: 93.25%
Validation Acc: 88.32%
Test Accuracy: 90.05%


In [ ]:
# Download real (unedited) videos
!wget https://github.com/adobe-research/VideoSham-dataset/releases/download/v1.0/Unedited-Part1.zip
!wget https://github.com/adobe-research/VideoSham-dataset/releases/download/v1.0/Unedited-Part2.zip

# Download fake (processed) videos
!wget https://github.com/adobe-research/VideoSham-dataset/releases/download/v1.0/Processed-Part1.zip
!wget https://github.com/adobe-research/VideoSham-dataset/releases/download/v1.0/Processed-Part2.zip

# Unzip into working directory
!unzip -q Unedited-Part1.zip -d videos/unedited_part1/
!unzip -q Unedited-Part2.zip -d videos/unedited_part2/
!unzip -q Processed-Part1.zip -d videos/processed_part1/
!unzip -q Processed-Part2.zip -d videos/processed_part2/



--2025-08-31 13:32:11--  https://github.com/adobe-research/VideoSham-dataset/releases/download/v1.0/Unedited-Part1.zip
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/528544789/d664624c-e582-491c-8728-81c922a2929b?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-08-31T14%3A09%3A49Z&rscd=attachment%3B+filename%3DUnedited-Part1.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-08-31T13%3A09%3A18Z&ske=2025-08-31T14%3A09%3A49Z&sks=b&skv=2018-11-09&sig=OMBMlWPnkIBfwgo2jMoaOhtIyj6eu2RxqNQEGDHyM48%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc1NjY0NzQzMSwibmJmIjoxNzU2NjQ3MTMxLCJwYXRoIjoicmVsZWFzZWFzc2V

In [21]:
# Create the folders
!mkdir -p videos/unedited_part1
!mkdir -p videos/unedited_part2
!mkdir -p videos/processed_part1
!mkdir -p videos/processed_part2
!ls videos/unedited_part1 | head
!ls videos/unedited_part2 | head
!ls videos/processed_part1 | head
!ls videos/processed_part2 | head



replace videos/unedited_part1/061_unedited.mp4? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C
[Unedited-Part2.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of Unedited-Part2.zip or
        Unedited-Part2.zip.zip, and cannot find Unedited-Part2.zip.ZIP, period.
[Processed-Part1.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of Processed-Part1.zip or
        Processed-Part1.zip.zip, and cannot find Processed-Part1.zip.ZIP, period.
[Processed-Part2.zip]
  End-of-central-directory signat

In [24]:
import os, shutil

# Source paths
src_real = ["videos/unedited_part1", "videos/unedited_part2"]
src_fake = ["videos/processed_part1", "videos/processed_part2"]

# Destination paths
base_dir = "VideoSham_dataset"
real_dir = os.path.join(base_dir, "real")
fake_dir = os.path.join(base_dir, "fake")

os.makedirs(real_dir, exist_ok=True)
os.makedirs(fake_dir, exist_ok=True)

# Move all videos into class folders
for folder in src_real:
    for f in os.listdir(folder):
        shutil.move(os.path.join(folder, f), os.path.join(real_dir, f))

for folder in src_fake:
    for f in os.listdir(folder):
        shutil.move(os.path.join(folder, f), os.path.join(fake_dir, f))

print("Dataset organized ")
print("Real videos:", len(os.listdir(real_dir)))
print("Fake videos:", len(os.listdir(fake_dir)))


Dataset organized 
Real videos: 180
Fake videos: 0


In [ ]:
import glob, random

real_videos = [(path, "real") for path in glob.glob("VideoSham_dataset/real/*.mp4")]
fake_videos = [(path, "fake") for path in glob.glob("VideoSham_dataset/fake/*.mp4")]

all_videos = real_videos + fake_videos
random.shuffle(all_videos)

train_split = int(0.7 * len(all_videos))
val_split   = int(0.85 * len(all_videos))

train_videos = all_videos[:train_split]
val_videos   = all_videos[train_split:val_split]
test_videos  = all_videos[val_split:]

print(len(train_videos), len(val_videos), len(test_videos))


In [ ]:
import os, glob, random, shutil
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models
import torch.optim as optim

# ==================================================
# 1. CONFIG
# ==================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

frames_per_video = 8
batch_size = 4
epochs = 10
base_dir = "VideoSham_dataset"

# ==================================================
# 2. DATASET
# ==================================================
transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor()
])

class VideoDataset(Dataset):
    def __init__(self, samples, transform=None, frames_per_video=8):
        self.samples = samples
        self.transform = transform
        self.frames_per_video = frames_per_video

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        cap = cv2.VideoCapture(video_path)
        frames = []
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames == 0:
            cap.release()
            raise ValueError(f"No frames found in video: {video_path}")

        if total_frames < self.frames_per_video:
            chosen_indices = sorted(random.choices(range(total_frames), k=self.frames_per_video))
        else:
            chosen_indices = sorted(random.sample(range(total_frames), self.frames_per_video))

        for f in chosen_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, f)
            ret, frame = cap.read()
            if not ret:
                continue
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            if self.transform:
                frame = self.transform(frame)
            frames.append(frame)

        cap.release()

        while len(frames) < self.frames_per_video:
            frames.append(frames[-1].clone())

        label = 0 if label == "real" else 1
        return torch.stack(frames), label

# ==================================================
# 3. TRAIN/VAL/TEST SPLIT
# ==================================================
real_videos = [(path, "real") for path in glob.glob(f"{base_dir}/real/*.mp4")]
fake_videos = [(path, "fake") for path in glob.glob(f"{base_dir}/fake/*.mp4")]
all_videos = real_videos + fake_videos
random.shuffle(all_videos)

train_split = int(0.7 * len(all_videos))
val_split   = int(0.85 * len(all_videos))

train_videos = all_videos[:train_split]
val_videos   = all_videos[train_split:val_split]
test_videos  = all_videos[val_split:]

print("Train:", len(train_videos), "Val:", len(val_videos), "Test:", len(test_videos))

train_loader = DataLoader(VideoDataset(train_videos, transform=transform, frames_per_video=frames_per_video),
                          batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(VideoDataset(val_videos, transform=transform, frames_per_video=frames_per_video),
                          batch_size=batch_size)
test_loader  = DataLoader(VideoDataset(test_videos, transform=transform, frames_per_video=frames_per_video),
                          batch_size=batch_size)

# ==================================================
# 4. MODEL (ResNet50)
# ==================================================
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# ==================================================
# 5. TRAINING LOOP
# ==================================================
for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0, 0, 0

    for frames, labels in train_loader:
        frames = frames.to(device)  # [B, F, C, H, W]
        labels = labels.to(device)

        # Average frames (temporal pooling)
        frames = frames.mean(dim=1)

        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss/len(train_loader):.4f} Acc: {train_acc:.4f}")

# ==================================================
# 6. SAVE MODEL
# ==================================================
torch.save(model.state_dict(), "resnet50_videosham.pth")
print("Model saved ")


In [ ]:
import torch

# Dummy input: one batch, 3 channels, 224x224 image
example_input = torch.randn(1, 3, 224, 224).to(device)

# TorchScript
traced_model = torch.jit.trace(model, example_input)
torch.jit.save(traced_model, "resnet50_videosham_torchscript.pt")

# ONNX
torch.onnx.export(
    model, example_input, "resnet50_videosham.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=11
)
print(" Model exported: TorchScript & ONNX")


In [ ]:
import zipfile
import os

# Path where models are stored
model_dir = "models"
zip_path = "models.zip"

# Create a zip archive
with zipfile.ZipFile(zip_path, "w") as z:
    for root, dirs, files in os.walk(model_dir):
        for file in files:
            filepath = os.path.join(root, file)
            z.write(filepath, os.path.relpath(filepath, model_dir))

print("Models zipped successfully:", zip_path)


In [ ]:
import zipfile
import os

# Path where models are stored
model_dir = "models"
zip_path = "all_models.zip"

# Create a zip archive
with zipfile.ZipFile(zip_path, "w") as z:
    for root, dirs, files in os.walk(model_dir):
        for file in files:
            filepath = os.path.join(root, file)
            z.write(filepath, os.path.relpath(filepath, model_dir))

print("All models zipped successfully:", zip_path)
